In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import os
import warnings
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import gc

warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Custom Dataset for Fruits-360 ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((os.path.join(class_dir, img_name), 
                                           self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label

def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with augmentation"""
    
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dir = os.path.join(data_root, 'Training')
    test_dir = os.path.join(data_root, 'Test')
    
    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset = FruitsDataset(test_dir, transform=transform_test)
    
    return trainset, testset, trainset.classes

# ================== FFT Conversion - Magnitude Only ==================

def spatial_to_frequency_magnitude(images):
    """Convert spatial domain images to frequency domain magnitude only"""
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    
    # Log-scale normalization for magnitude
    eps = torch.mean(freq_magnitude) * 0.01
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    freq_magnitude_normalized = (freq_magnitude_log - freq_magnitude_log.mean()) / (freq_magnitude_log.std() + 1e-8)
    
    return freq_magnitude_normalized

# ================== Frequency Domain Dataset - Magnitude Only ==================

class FrequencyMagnitudeDataset(Dataset):
    """Dataset for frequency domain magnitude only"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        
        # Convert to frequency domain (magnitude only)
        with torch.no_grad():
            freq_magnitude = spatial_to_frequency_magnitude(image.unsqueeze(0))
            freq_magnitude = freq_magnitude.squeeze(0)
        
        return freq_magnitude, label

# ================== ResNet50 Model for Magnitude Only (3 channels) ==================

class FrequencyMagnitudeCNN(nn.Module):
    """ResNet50-based model for frequency domain magnitude (3 channels)"""
    
    def __init__(self, num_classes, dropout_rate=0.5):
        super(FrequencyMagnitudeCNN, self).__init__()
        
        self.model = models.resnet50(pretrained=True)
        
        # Keep original conv1 for 3-channel input (magnitude only)
        # No modification needed as magnitude has 3 channels (RGB)
        
        # Enhanced classifier head
        num_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ================== Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, weight_decay=1e-4):
    """Train the frequency magnitude CNN model"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    pretrained_params = []
    new_params = []
    
    for name, param in model.named_parameters():
        if 'model.fc' in name:
            new_params.append(param)
        else:
            pretrained_params.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.1},
        {'params': new_params, 'lr': lr}
    ], weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for i, (freq_images, labels) in enumerate(train_pbar):
            freq_images, labels = freq_images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(freq_images)
                    loss = criterion(outputs, labels)
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
            
            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        scheduler.step()
        
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
            for freq_images, labels in val_pbar:
                freq_images, labels = freq_images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                
                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        outputs = model(freq_images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(freq_images)
                    loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100 * correct / total:.2f}%'
                })
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}')
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Visualization Functions ==================

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(y_true, y_pred, classes, normalize=False):
    """Plot confusion matrix"""
    
    cm = confusion_matrix(y_true, y_pred)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2f'
        title = 'Normalized Confusion Matrix'
    else:
        fmt = 'd'
        title = 'Confusion Matrix'
    
    plt.figure(figsize=(20, 18))
    
    # For large number of classes, don't show all labels
    if len(classes) > 50:
        sns.heatmap(cm, annot=False, fmt=fmt, cmap='Blues', 
                   cbar_kws={'label': 'Count' if not normalize else 'Proportion'})
        plt.title(f'{title}\n({len(classes)} classes - labels hidden for clarity)', 
                 fontsize=16, fontweight='bold', pad=20)
    else:
        sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues', 
                   xticklabels=classes, yticklabels=classes,
                   cbar_kws={'label': 'Count' if not normalize else 'Proportion'})
        plt.title(title, fontsize=16, fontweight='bold', pad=20)
        plt.xticks(rotation=90, ha='right')
        plt.yticks(rotation=0)
    
    plt.xlabel('Predicted Label', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ================== Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Frequency Domain CNN (Magnitude Only) - ResNet50 on Fruits-360")
    print("="*80)
    
    # Dataset path - MODIFY THIS PATH
    data_root = r'C:\Users\CSE_SDPL\Downloads\data\fruits-360_100x100\fruits-360'
    
    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Please modify the 'data_root' variable to point to your dataset location.")
        return
    
    print("\n[Step 1] Loading Fruits-360 dataset...")
    try:
        trainset, testset, classes = load_fruits_dataset(data_root)
        print(f"Number of classes: {len(classes)}")
    except Exception as e:
        print(f"ERROR loading dataset: {e}")
        return
    
    print("\n[Step 2] Splitting training set into train/validation...")
    train_size = int(0.85 * len(trainset))
    val_size = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"Training samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test samples: {len(testset)}")
    
    print("\n[Step 3] Converting to frequency domain (magnitude only)...")
    freq_train_dataset = FrequencyMagnitudeDataset(train_subset)
    freq_val_dataset = FrequencyMagnitudeDataset(val_subset)
    freq_test_dataset = FrequencyMagnitudeDataset(testset)
    
    batch_size = 128
    num_workers = 4 if os.name != 'nt' else 0
    
    print(f"Batch size: {batch_size}")
    print(f"Num workers: {num_workers}")
    
    train_loader = DataLoader(
        freq_train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    val_loader = DataLoader(
        freq_val_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    test_loader = DataLoader(
        freq_test_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )
    
    print("\n[Step 4] Initializing ResNet50 model for frequency magnitude...")
    model = FrequencyMagnitudeCNN(num_classes=len(classes), dropout_rate=0.5).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    
    print("\n[Step 5] Training model on frequency magnitude data...")
    
    try:
        train_losses, val_losses, train_accuracies, val_accuracies = train_model(
            model, train_loader, val_loader, 
            epochs=40,
            lr=0.001,
            weight_decay=5e-4
        )
    except Exception as e:
        print(f"\nERROR during training: {e}")
        import traceback
        traceback.print_exc()
        return
    
    print("\n[Step 6] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    print("\n[Step 7] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for freq_images, labels in test_pbar:
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            test_pbar.set_postfix({'acc': f'{100 * correct / total:.2f}%'})
    
    test_accuracy = 100 * correct / total
    print(f"\n{'='*80}")
    print(f"FINAL TEST ACCURACY: {test_accuracy:.2f}%")
    print(f"{'='*80}\n")
    
    print("\n[Step 8] Generating confusion matrix...")
    plot_confusion_matrix(all_labels, all_predictions, classes, normalize=False)
    
    print("\n[Step 9] Generating normalized confusion matrix...")
    plot_confusion_matrix(all_labels, all_predictions, classes, normalize=True)
    
    print("\n[Step 10] Classification Report:")
    print("="*80)
    # For many classes, show a summary report
    report = classification_report(all_labels, all_predictions, 
                                   target_names=classes, 
                                   digits=4,
                                   zero_division=0)
    print(report)
    print("="*80)
    
    print("\n[Step 11] Saving trained model...")
    try:
        torch.save({
            'model_state_dict': model.state_dict(),
            'test_accuracy': test_accuracy,
            'classes': classes,
            'num_classes': len(classes),
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accuracies': train_accuracies,
            'val_accuracies': val_accuracies
        }, 'fruits_magnitude_only_resnet50.pth')
        print("Model saved as 'fruits_magnitude_only_resnet50.pth'")
    except Exception as e:
        print(f"ERROR saving model: {e}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("\n" + "="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print(f"Total Classes: {len(classes)}")
    print("="*80)

if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support, cohen_kappa_score
from tqdm.auto import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ================== Dataset and Model Classes ==================

class FruitsDataset:
    """Custom dataset for Fruits-360"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((os.path.join(class_dir, img_name), 
                                           self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label

class FrequencyDomainDataset:
    """Custom dataset for frequency domain representations (magnitude-only)"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        
        with torch.no_grad():
            freq_magnitude = spatial_to_frequency_magnitude(image.unsqueeze(0))
            freq_magnitude = freq_magnitude.squeeze(0)
        
        return freq_magnitude, label

class MagnitudeOnlyResNet50(nn.Module):
    """ResNet50-based model for magnitude-only frequency domain"""
    
    def __init__(self, num_classes, dropout_rate=0.5):
        super(MagnitudeOnlyResNet50, self).__init__()
        
        self.model = models.resnet50(pretrained=False)
        # Modify first conv layer for 3-channel magnitude input
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        num_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.model(x)

def spatial_to_frequency_magnitude(images):
    """Convert spatial domain images to frequency domain magnitude only"""
    freq_complex = torch.fft.fft2(images, dim=(-2, -1))
    freq_complex = torch.fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    
    eps = torch.mean(freq_magnitude) * 0.01
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    freq_magnitude_normalized = (freq_magnitude_log - freq_magnitude_log.mean()) / (freq_magnitude_log.std() + 1e-8)
    
    return freq_magnitude_normalized

# ================== Evaluation and Visualization Functions ==================

def evaluate_model(model, test_loader, classes):
    """Evaluate model and collect predictions"""
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    print("\nEvaluating model on test set...")
    with torch.no_grad():
        for freq_images, labels in tqdm(test_loader, desc="Testing"):
            freq_images = freq_images.to(device)
            outputs = model(freq_images)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    all_probabilities = np.array(all_probabilities)
    
    return all_predictions, all_labels, all_probabilities

def plot_confusion_matrix(y_true, y_pred, classes, figsize=(20, 18), save_path=None):
    """Plot confusion matrix with enhanced visualization"""
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate percentages
    cm_percentage = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Plot 1: Raw counts
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Count'}, ax=ax1)
    ax1.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax1.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax1.set_title('Confusion Matrix (Raw Counts)', fontsize=14, fontweight='bold')
    ax1.tick_params(axis='x', rotation=90, labelsize=8)
    ax1.tick_params(axis='y', rotation=0, labelsize=8)
    
    # Plot 2: Percentages
    sns.heatmap(cm_percentage, annot=False, fmt='.1f', cmap='YlOrRd',
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Percentage (%)'}, ax=ax2)
    ax2.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax2.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax2.set_title('Confusion Matrix (Percentage)', fontsize=14, fontweight='bold')
    ax2.tick_params(axis='x', rotation=90, labelsize=8)
    ax2.tick_params(axis='y', rotation=0, labelsize=8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Confusion matrix saved to: {save_path}")
    
    plt.show()
    
    return cm

def calculate_per_class_metrics(cm, class_idx):
    """Calculate sensitivity, specificity, and error rate for a specific class"""
    tp = cm[class_idx, class_idx]
    fn = np.sum(cm[class_idx, :]) - tp
    fp = np.sum(cm[:, class_idx]) - tp
    tn = np.sum(cm) - tp - fn - fp
    
    # Sensitivity (Recall/True Positive Rate)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    # Specificity (True Negative Rate)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    # Error Rate
    error_rate = (fp + fn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    
    return sensitivity, specificity, error_rate

def generate_comprehensive_classification_report(y_true, y_pred, classes, save_path=None):
    """Generate comprehensive classification report with all metrics including per-sample details"""
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    
    # Calculate overall metrics
    overall_accuracy = accuracy_score(y_true, y_pred)
    cohen_kappa = cohen_kappa_score(y_true, y_pred)
    
    # Build comprehensive report
    report_lines = []
    report_lines.append("="*120)
    report_lines.append("COMPREHENSIVE CLASSIFICATION REPORT")
    report_lines.append("Magnitude-Only ResNet50 - Fruits-360 Dataset")
    report_lines.append("="*120)
    report_lines.append("")
    
    # Overall metrics
    report_lines.append("OVERALL METRICS:")
    report_lines.append("-"*120)
    report_lines.append(f"Overall Accuracy:           {overall_accuracy:.6f} ({overall_accuracy*100:.4f}%)")
    report_lines.append(f"Cohen's Kappa Score:        {cohen_kappa:.6f}")
    report_lines.append(f"Total Test Samples:         {len(y_true)}")
    report_lines.append(f"Number of Classes:          {len(classes)}")
    report_lines.append(f"Correctly Classified:       {np.sum(y_true == y_pred)}")
    report_lines.append(f"Misclassified:              {np.sum(y_true != y_pred)}")
    report_lines.append("")
    
    # Per-class metrics header
    report_lines.append("="*120)
    report_lines.append("PER-CLASS METRICS:")
    report_lines.append("="*120)
    report_lines.append("")
    report_lines.append(f"{'Class Name':<30} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Sensitivity':<13} {'Specificity':<13} {'Error Rate':<12} {'Support':<10}")
    report_lines.append("-"*120)
    
    # Calculate and display per-class metrics
    for i, class_name in enumerate(classes):
        sensitivity, specificity, error_rate = calculate_per_class_metrics(cm, i)
        
        report_lines.append(
            f"{class_name:<30} "
            f"{precision[i]:<12.6f} "
            f"{recall[i]:<12.6f} "
            f"{f1[i]:<12.6f} "
            f"{sensitivity:<13.6f} "
            f"{specificity:<13.6f} "
            f"{error_rate:<12.6f} "
            f"{support[i]:<10}"
        )
    
    report_lines.append("-"*120)
    
    # Weighted averages
    weighted_precision = np.average(precision, weights=support)
    weighted_recall = np.average(recall, weights=support)
    weighted_f1 = np.average(f1, weights=support)
    
    # Calculate weighted sensitivity, specificity, and error rate
    weighted_sensitivity = 0
    weighted_specificity = 0
    weighted_error_rate = 0
    total_support = np.sum(support)
    
    for i in range(len(classes)):
        sensitivity, specificity, error_rate = calculate_per_class_metrics(cm, i)
        weighted_sensitivity += sensitivity * support[i] / total_support
        weighted_specificity += specificity * support[i] / total_support
        weighted_error_rate += error_rate * support[i] / total_support
    
    report_lines.append(
        f"{'WEIGHTED AVERAGE':<30} "
        f"{weighted_precision:<12.6f} "
        f"{weighted_recall:<12.6f} "
        f"{weighted_f1:<12.6f} "
        f"{weighted_sensitivity:<13.6f} "
        f"{weighted_specificity:<13.6f} "
        f"{weighted_error_rate:<12.6f} "
        f"{total_support:<10}"
    )
    
    # Macro averages
    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)
    
    macro_sensitivity = np.mean([calculate_per_class_metrics(cm, i)[0] for i in range(len(classes))])
    macro_specificity = np.mean([calculate_per_class_metrics(cm, i)[1] for i in range(len(classes))])
    macro_error_rate = np.mean([calculate_per_class_metrics(cm, i)[2] for i in range(len(classes))])
    
    report_lines.append(
        f"{'MACRO AVERAGE':<30} "
        f"{macro_precision:<12.6f} "
        f"{macro_recall:<12.6f} "
        f"{macro_f1:<12.6f} "
        f"{macro_sensitivity:<13.6f} "
        f"{macro_specificity:<13.6f} "
        f"{macro_error_rate:<12.6f} "
        f"{total_support:<10}"
    )
    
    report_lines.append("-"*120)
    report_lines.append("")
    
    # Per-sample predictions
    report_lines.append("="*120)
    report_lines.append("PER-SAMPLE PREDICTIONS:")
    report_lines.append("="*120)
    report_lines.append("")
    report_lines.append(f"{'Sample #':<10} {'True Label':<30} {'Predicted Label':<30} {'Result':<15}")
    report_lines.append("-"*120)
    
    for idx in range(len(y_true)):
        true_label = classes[y_true[idx]]
        pred_label = classes[y_pred[idx]]
        result = "CORRECT ✓" if y_true[idx] == y_pred[idx] else "INCORRECT ✗"
        
        report_lines.append(
            f"{idx+1:<10} "
            f"{true_label:<30} "
            f"{pred_label:<30} "
            f"{result:<15}"
        )
    
    report_lines.append("-"*120)
    report_lines.append("")
    report_lines.append("="*120)
    report_lines.append("END OF REPORT")
    report_lines.append("="*120)
    
    # Print to console
    report_text = "\n".join(report_lines)
    print(report_text)
    
    # Save to file
    if save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(report_text)
        print(f"\nComprehensive classification report saved to: {save_path}")
    
    return report_text

def generate_summary_metrics(y_true, y_pred, classes):
    """Generate summary statistics"""
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    cohen_kappa = cohen_kappa_score(y_true, y_pred)
    
    print("\n" + "="*80)
    print("SUMMARY METRICS")
    print("="*80)
    print(f"Overall Accuracy:        {accuracy*100:.2f}%")
    print(f"Cohen's Kappa Score:     {cohen_kappa:.4f}")
    print(f"Weighted Precision:      {precision:.4f}")
    print(f"Weighted Recall:         {recall:.4f}")
    print(f"Weighted F1-Score:       {f1:.4f}")
    print(f"Total Test Samples:      {len(y_true)}")
    print(f"Number of Classes:       {len(classes)}")
    print("="*80)

# ================== Main Execution ==================

def main():
    print("="*80)
    print("CONFUSION MATRIX & CLASSIFICATION REPORT GENERATOR")
    print("Magnitude-Only ResNet50 - Fruits-360 Dataset")
    print("="*80)
    
    # MODIFY THESE PATHS
    data_root = r'C:\Users\CSE_SDPL\Downloads\data\fruits-360_100x100\fruits-360'
    model_path = 'fruits_magnitude_only_resnet50.pth'
    
    # Check paths
    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        return
    
    if not os.path.exists(model_path):
        print(f"\nERROR: Model file not found: {model_path}")
        print("Please train the model first using the main script.")
        return
    
    # Load test dataset
    print("\n[Step 1] Loading test dataset...")
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_dir = os.path.join(data_root, 'Test')
    testset = FruitsDataset(test_dir, transform=transform_test)
    classes = testset.classes
    
    print(f"Number of classes: {len(classes)}")
    print(f"Test samples: {len(testset)}")
    
    # Convert to frequency domain (magnitude-only)
    print("\n[Step 2] Converting to frequency domain (magnitude-only)...")
    freq_test_dataset = FrequencyDomainDataset(testset)
    test_loader = DataLoader(freq_test_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    # Load model
    print("\n[Step 3] Loading trained model...")
    checkpoint = torch.load(model_path, map_location=device)
    
    model = MagnitudeOnlyResNet50(num_classes=len(classes), dropout_rate=0.5).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    print(f"Model loaded successfully!")
    print(f"Saved test accuracy: {checkpoint.get('test_accuracy', 'N/A'):.2f}%")
    
    # Evaluate model
    print("\n[Step 4] Evaluating model and collecting predictions...")
    y_pred, y_true, y_proba = evaluate_model(model, test_loader, classes)
    
    # Generate summary metrics
    print("\n[Step 5] Generating summary metrics...")
    generate_summary_metrics(y_true, y_pred, classes)
    
    # Generate comprehensive classification report with all metrics
    print("\n[Step 6] Generating comprehensive classification report...")
    print("(This may take a moment for large datasets...)")
    generate_comprehensive_classification_report(y_true, y_pred, classes, save_path='classification_report_mag.txt')
    
    # Generate confusion matrix
    print("\n[Step 7] Generating confusion matrix...")
    cm = plot_confusion_matrix(y_true, y_pred, classes, 
                               figsize=(20, 18), 
                               save_path='confusion_matrix_mag.png')
    
    print("\n" + "="*80)
    print("EVALUATION COMPLETED SUCCESSFULLY!")
    print("="*80)
    print("\nGenerated files:")
    print("  ✓ classification_report.txt  - Comprehensive report with all metrics and per-sample predictions")
    print("  ✓ confusion_matrix.png       - Visual confusion matrix (counts and percentages)")
    print("="*80)
    print("\nThe classification_report.txt contains:")
    print("  • Overall Accuracy & Cohen's Kappa Score")
    print("  • Per-Class Metrics: Precision, Recall, F1-Score, Sensitivity, Specificity, Error Rate, Support")
    print("  • Weighted & Macro Averages")
    print("  • Per-Sample Predictions (True Label, Predicted Label, Result)")
    print("="*80)

if __name__ == "__main__":
    main()